# EV Charging Pattern Analysis: Customer Segmentation with K-Means and DBSCAN

**Goal:** Explore EV charging behavior, identify meaningful segments, and translate those segments into operational recommendations. This project improves the original case-study notebook while retaining **K-Means** and **DBSCAN**.

**Dataset:** [Electric Vehicle Charging Patterns (Kaggle)](https://www.kaggle.com/datasets/valakhorasani/electric-vehicle-charging-patterns). Download the CSV separately and place it in `data/ev_charging_patterns.csv` (subject to the dataset license).

**Important:** All charts, cluster counts, scores, and interpretations below are generated when you run the notebook. No results are claimed in advance.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

RANDOM_STATE = 7
sns.set_theme(style="whitegrid")
DATA_PATH = Path("data/ev_charging_patterns.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Download the Kaggle CSV and save it at: {DATA_PATH.resolve()}")

## 2. Load and audit the dataset

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Original shape:", raw.shape)
display(raw.head())
display(pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum(), "missing_pct": raw.isna().mean().mul(100).round(2), "unique": raw.nunique(dropna=True)}))
print("Duplicate rows:", raw.duplicated().sum())

## 3. Exploratory data analysis

Use the **original unscaled units** for descriptive plots. These charts describe the dataset; they do not by themselves establish causation.

In [ ]:
numeric_raw = raw.select_dtypes(include="number")
if not numeric_raw.empty:
    display(numeric_raw.describe().T)
    numeric_raw.hist(figsize=(15, 10), bins=25)
    plt.suptitle("Numeric feature distributions")
    plt.tight_layout()
    plt.show()

for col in ["Charger Type", "User Type", "Vehicle Model"]:
    if col in raw:
        plt.figure(figsize=(8, 3))
        raw[col].value_counts(dropna=False).head(12).plot(kind="bar")
        plt.title(f"Distribution: {col}")
        plt.ylabel("Records")
        plt.tight_layout()
        plt.show()

In [ ]:
if numeric_raw.shape[1] >= 2:
    plt.figure(figsize=(11, 8))
    sns.heatmap(numeric_raw.corr(), cmap="coolwarm", center=0)
    plt.title("Numeric feature correlation (exploratory)")
    plt.tight_layout()
    plt.show()

## 4. Prepare modeling features

- Remove ID fields, which represent identifiers rather than behavior.
- Remove raw timestamps as in the original notebook. A future extension could engineer hour-of-day and session duration from them.
- Treat **User Type** and **Charger Type** as nominal categories for clustering: assigning integer ranks can impose artificial distance assumptions.
- Median-impute numeric values, mode-impute categorical values, one-hot encode categories, then standardize numeric columns.
- Remove duplicates **before** transformation.

**Outlier decision:** Do not automatically IQR-cap every numeric variable. Extreme charging behavior may be the very signal DBSCAN should detect. Review distributions and apply domain-justified capping only if needed.

In [ ]:
drop_cols = ["User ID", "Charging Station ID", "Charging Start Time", "Charging End Time"]
features = raw.drop(columns=drop_cols, errors="ignore").drop_duplicates().reset_index(drop=True)
all_missing = features.columns[features.isna().all()].tolist()
if all_missing:
    print("Dropping all-missing columns:", all_missing)
    features = features.drop(columns=all_missing)
num_cols = features.select_dtypes(include="number").columns.tolist()
cat_cols = features.select_dtypes(exclude="number").columns.tolist()
print("Rows / features:", features.shape, "Numeric:", len(num_cols), "Categorical:", len(cat_cols))
if not num_cols and not cat_cols:
    raise ValueError("No usable features found")
transformers = []
if num_cols:
    transformers.append(("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
X = preprocessor.fit_transform(features)
X = np.asarray(X, dtype=float)
print("Modeling matrix:", X.shape, "Finite values:", np.isfinite(X).all())

## 5. K-Means: compare candidate cluster counts

K-Means is fitted to the **same preprocessed feature matrix** used for DBSCAN. Because clustering is unsupervised, we fit and evaluate on the full dataset for exploratory segmentation. For production use, validate segment stability on independent or resampled data.

In [ ]:
max_k = min(10, len(X) - 1)
if max_k < 2:
    raise ValueError("At least 3 records are needed for this analysis")
km_rows = []
for k in range(2, max_k + 1):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(X)
    km_rows.append({"k": k, "inertia": model.inertia_, "silhouette": silhouette_score(X, labels), "davies_bouldin": davies_bouldin_score(X, labels)})
km_scores = pd.DataFrame(km_rows)
display(km_scores.round(4))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(km_scores.k, km_scores.inertia, marker="o")
axes[0].set(title="Elbow: inertia", xlabel="Number of clusters (k)", ylabel="Inertia")
axes[1].plot(km_scores.k, km_scores.silhouette, marker="o")
axes[1].set(title="Silhouette by k", xlabel="Number of clusters (k)", ylabel="Silhouette")
plt.tight_layout()
plt.show()

### Choose K

The original notebook used **k=3** without a documented justification. Here, the best silhouette score is used as a reproducible *starting point*. Review the elbow curve, cluster sizes, and interpretability before deciding whether to override this choice.

In [ ]:
best_k = int(km_scores.loc[km_scores.silhouette.idxmax(), "k"])
print("Silhouette-selected k:", best_k)
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
km_labels = kmeans.fit_predict(X)
print("K-Means cluster sizes:")
display(pd.Series(km_labels, name="cluster").value_counts().sort_index().rename("records").to_frame())

## 6. DBSCAN: explore neighborhood distances and parameter sensitivity

A 2-neighbor distance plot is not sufficient evidence that `eps=5.4` is universally appropriate. Try several `eps` values and use a more meaningful `min_samples` starting point; evaluate only configurations with at least two non-noise clusters.

In [ ]:
min_samples = max(5, min(2 * X.shape[1], len(X) - 1))
neighbor_distances, _ = NearestNeighbors(n_neighbors=min_samples).fit(X).kneighbors(X)
k_distances = np.sort(neighbor_distances[:, -1])
plt.figure(figsize=(9, 4))
plt.plot(k_distances)
plt.title(f"Sorted distance to {min_samples}th nearest neighbor")
plt.xlabel("Observations sorted by distance")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()
# Percentiles are exploratory candidates, not an automatic knee detector.
eps_candidates = np.unique(np.quantile(k_distances, [0.60, 0.70, 0.80, 0.90, 0.95]).round(3))
db_rows = []
for eps in eps_candidates:
    labels = DBSCAN(eps=float(eps), min_samples=min_samples).fit_predict(X)
    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    score = silhouette_score(X[mask], labels[mask]) if n_clusters >= 2 and mask.sum() > n_clusters else np.nan
    db_rows.append({"eps": eps, "min_samples": min_samples, "clusters": n_clusters, "noise_pct": 100 * (~mask).mean(), "silhouette_non_noise": score})
db_scores = pd.DataFrame(db_rows)
display(db_scores.round(3))

In [ ]:
valid = db_scores.dropna(subset=["silhouette_non_noise"])
if valid.empty:
    print("No tested DBSCAN setting produced at least two non-noise clusters. Inspect the distance plot and expand the parameter search.")
    db_labels = None
else:
    chosen = valid.sort_values(["silhouette_non_noise", "noise_pct"], ascending=[False, True]).iloc[0]
    db_labels = DBSCAN(eps=float(chosen.eps), min_samples=min_samples).fit_predict(X)
    print("Chosen DBSCAN parameters (exploratory):", {"eps": float(chosen.eps), "min_samples": min_samples})
    display(pd.Series(db_labels, name="cluster").value_counts().sort_index().rename("records").to_frame())
    print("Caution: non-noise silhouette excludes points labeled -1; compare coverage and noise rate as well.")

## 7. Visualize clusters using PCA

PCA is used **only for 2D visualization**, not for training the clustering algorithms. Apparent overlap in 2D does not necessarily imply overlap in the full feature space.

In [ ]:
if X.shape[1] >= 2:
    projected = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
    fig, axes = plt.subplots(1, 2 if db_labels is not None else 1, figsize=(13 if db_labels is not None else 7, 5))
    axes = np.atleast_1d(axes)
    axes[0].scatter(projected[:, 0], projected[:, 1], c=km_labels, cmap="tab10", s=15, alpha=0.7)
    axes[0].set_title("K-Means clusters (PCA projection)")
    if db_labels is not None:
        axes[1].scatter(projected[:, 0], projected[:, 1], c=db_labels, cmap="tab10", s=15, alpha=0.7)
        axes[1].set_title("DBSCAN clusters; -1 = noise")
    for ax in axes:
        ax.set(xlabel="PC1", ylabel="PC2")
    plt.tight_layout()
    plt.show()

## 8. Cluster profiles and business interpretation

Profiles are calculated on **original feature values**, not scaled values. Compare cluster medians and categorical composition to assign meaningful segment names. Cluster IDs are arbitrary; avoid naming them until inspecting these outputs.

In [ ]:
profile = features.copy()
profile["kmeans_cluster"] = km_labels
if num_cols:
    medians = profile.groupby("kmeans_cluster")[num_cols].median(numeric_only=True)
    print("K-Means cluster medians:")
    display(medians.round(2))
    overall = profile[num_cols].median(numeric_only=True)
    print("Cluster median difference from overall median:")
    display(medians.sub(overall).round(2))
for col in ["Charger Type", "User Type"]:
    if col in profile:
        print(f"Distribution of {col} by K-Means cluster (% within cluster):")
        display((pd.crosstab(profile.kmeans_cluster, profile[col], normalize="index") * 100).round(1))
if db_labels is not None:
    db_profile = features.copy()
    db_profile["dbscan_cluster"] = db_labels
    if num_cols:
        print("DBSCAN cluster medians (including noise=-1):")
        display(db_profile.groupby("dbscan_cluster")[num_cols].median(numeric_only=True).round(2))

## 9. Findings and recommendations — complete after running

Fill these in using the computed tables above, rather than assuming any outcome:

1. **Dataset:** ___ records after deduplication, ___ model features after encoding.
2. **K-Means:** selected k=___, silhouette=___; cluster sizes=___.
3. **DBSCAN:** eps=___, min_samples=___, clusters=___, noise=___% (or explain why no useful configuration emerged).
4. **Behavioral interpretation:** Which clusters differ most in charging duration, energy use, cost, charger type, or user type? Cite the profile table.
5. **Business actions:** For example, if supported by the data, evaluate whether high-demand segments justify additional fast chargers, or whether longer sessions suggest scheduling or pricing changes.
6. **Limitations:** Dataset provenance and representativeness; encoding and distance sensitivity; no causal inference; clusters require stability checks and stakeholder validation.

**Note:** Clustering can describe behavioral patterns, but does not demonstrate that a particular operational change will improve business outcomes.

## 10. Optional next steps

- Engineer session duration and charging start hour from the timestamp fields.
- Compare robust scaling and categorical-aware distance methods.
- Check cluster stability across random seeds, bootstrap samples, and feature subsets.
- Build a Power BI dashboard with cluster counts, usage patterns, and charging KPIs.